# tmpl

> The template layer shared by claudedojo and codexdojo

In [ ]:
#| default_exp tmpl

#| export
The claudedojo, codexdojo, and ipyaidojo launchers are mirror images: each gates a clean dojo round, stores it beside its metadata, and splices it into sessions at launch time. They differ only in the record shape their host uses (Claude Code transcript records, Codex Responses items, an ipyai session dialog), in transport, and in the bootstrap that opens the template - the host's own doc() reads, answered to the first of two prompts. Everything representation-independent lives here instead of three times over there: the shared prompts and gate phrases, the completion-id receipt, the store layout, the launch-time loader and doc-state seeding, the content gates, the captured dialog's structural boundaries, and the assembly of a host's bootstrap with the shared round. A backend contributes only its bootstrap dialog, its extractors, its transport, and its capture child.

In [ ]:
#| export
import asyncio, json, os, re, subprocess, sys, tempfile, tomllib
from importlib.resources import files
from fastcore.utils import *
from fastcore.script import call_parse
from clikernel.cli import _MARKER as _CLIK_MARKER
from aidialog.ipynb import read_ipynb, write_ipynb
from aidialog.hist import reply2dlg, dlg2reply, _parse_call
from aidialog.dialog import Dialog, code_output, prompt_output
from llmdojo.rules import _state_root, _docnames

In [ ]:
from fastcore.test import *
import tempfile, shutil

## The shared script

Whichever host plays the round, the words around it are the same: the two user prompts that open a template (bootstrap, then the dojo), the prompt used when re-splicing after a compaction, the capture script a scripted child follows, and the script phrases that must never leak into a template's visible text. The two packaged dialogs are the review artifacts a build starts from: the shared round, and the clikernel hosts' bootstrap (ipyai keeps its own beside its backend).

In [ ]:
#| export
BOOT_PROMPT = "Bootstrap and tell me when you're ready."
TMPL_PROMPT = "Complete the dojo and tell me when you're ready."
APPEND_PROMPT = "We've compacted - bootstrap again and tell me when you're ready."
GATE_FORBID = ('recording a demo', 'these notes')
CAPTURE_SCRIPT = (files('llmdojo')/'dojo_data'/'capture_prompt.md').read_text()
ROUND_PATH = files('llmdojo')/'dojo_data'/'dojo_round.ipynb'   # the shared round: one prompt, its reply the worked dojo
CLIK_BOOT = files('llmdojo')/'dojo_data'/'clik_boot.ipynb'     # the clikernel hosts' bootstrap: one prompt, its reply the startup's doc() reads

## Completion receipts

A clean score prints a four-hex-character completion id, and that receipt is what launch-time registration honors. `find_cid` reads it from any iterable of record texts, so each backend passes its own text extractor's output. The dojo version rides along with every stored receipt: a version mismatch is how a stale template announces itself.

In [ ]:
#| export
def find_cid(
    txts, # Record texts, in order
):
    "The completion id receipt in a clean round's texts"
    return first(m[1] for t in txts if (m := re.search(r'Completion id: ([0-9a-f]{4})', t)))

def _dojo_v():
    from llmdojo.dojo import dojo_version
    return dojo_version()

In [ ]:
test_eq(find_cid(['strokes 8 = 8, par 8', 'Clean round. Completion id: ab12 - keep this id.']), 'ab12')
test_eq(find_cid(['no receipt here']), None)

## The template store

A store is one directory holding the round in the backend's native shape (`template.jsonl`) and the metadata needed to splice it (`meta.json`): the completion id, the dojo version it was played under, and the names `doc()`'d during the round. Items are stored as-is: the store neither knows nor cares whether a line is a Claude transcript record or a Codex Responses item.

In [ ]:
#| export
def save_store(
    d, # Store dir
    items, # Native template items or records
    cid, # The round's completion id receipt
    doced=None, # Names doc()'d during the baked round; the current conversation's doc-state if None
):
    "Write a template and its metadata to the store at `d`"
    if doced is None:
        from llmdojo.rules import doced as _cur
        doced = _cur()
    d = Path(d)
    d.mkdir(parents=True, exist_ok=True)
    (d/'template.jsonl').write_text(''.join(json.dumps(obj2dict(x))+'\n' for x in items))
    (d/'meta.json').write_text(json.dumps(dict(cid=cid, v=_dojo_v(), doced=list(doced))))

def load_store(
    d, # Store dir
):
    "The stored native items and metadata at `d`"
    d = Path(d)
    return (d/'template.jsonl').read_jsonl(), json.loads((d/'meta.json').read_text())

In [ ]:
items = [dict(role='user', content='hi'), dict(role='assistant', content='Completion id: ab12')]
store = Path(tempfile.mkdtemp())
save_store(store, items, 'ab12', doced=['rg','view_nb'])
back,meta = load_store(store)
test_eq(list(back), items)
test_eq(meta, dict(cid='ab12', v=_dojo_v(), doced=['rg','view_nb']))
shutil.rmtree(store)
meta

`load_reg` is the launch-time loader both backends wrap. The stores are package data - compiled beside the canonical dialog by `dojobuild` and shipped with the code - so there is nothing per-user to build or invalidate: updating llmdojo updates the template, and first launch on a fresh machine is a plain read. The loader warns on stderr when the packaged store's version doesn't match the installed tooling (a maintainer tripwire: the version was bumped without rerunning `dojobuild`; stdout is reserved - launchers print session ids for `$(...)` substitution), and registers the completion id so `dojo_start(cid)` honors the baked round. Registration writes machine-global state, so the backends' launch tests exercise it under a redirected state dir; there is no live example here.

In [ ]:
#| export
def load_reg(
    d, # Store dir; `default` if None
    default, # The backend's packaged store dir
    prog, # The backend's program name, for the skew warning
):
    "Load a template store, warn on version skew, and register its completion id"
    items,meta = load_store(Path(d or default))
    if meta['v'] != _dojo_v(): print(f"{prog}: template built under {meta['v']} but installed tooling is {_dojo_v()}; the baked round will not validate. Rebuild with dojobuild.", file=sys.stderr)
    from llmdojo.dojo import register_completion
    register_completion(meta['cid'], meta['v'])
    return items,meta

Every launch path also seeds doc-state, so the spliced round's claim to have read the docs is true on disk as well as in context. `merge=True` is for appending to a session that may already hold doc-state of its own (though after a compaction the hook has usually truncated it).

In [ ]:
#| export
def _seed_doced(sid, names, merge=False):
    "Make the baked round's doc-state true for conversation `sid`"
    d = _state_root()/'doced'
    d.mkdir(parents=True, exist_ok=True)
    f = d/f'{sid}.json'
    if merge and f.exists(): names = set(names) | set(json.loads(f.read_text()))
    f.write_text(json.dumps(sorted(names)))

## Content gates

A template must show exactly one round, dealt once and scored clean on the first try, with none of the capture script's phrasing leaking into the visible reply. These checks only need the round's kernel cell sources, its tool output texts, and the visible assistant text, so they are shared; representation-level checks (Claude's `is_error` results, Codex's orphaned call ids) stay with each backend's `is_clean`, which appends them to these.

In [ ]:
#| export
_START_RE = r'^dojo_start\(\s*\)\s*$'

def round_gates(
    cells, # The round's kernel cell sources, in order
    outs, # Its tool output texts
    visible, # Its visible assistant text, joined
    forbid=GATE_FORBID, # Phrases that must not appear in visible text
):
    "The content problems that disqualify a round from becoming a template, whichever host played it"
    probs = []
    if (n := sum(bool(re.match(_START_RE, c)) for c in cells)) != 1: probs.append(f'{n} dojo_start calls')
    if (n := sum(bool(re.match(r'^dojo_score\(', c)) for c in cells)) != 1: probs.append(f'{n} dojo_score calls (a clean round scores once, first try)')
    if any(re.match(r'^dojo_(?:redo|resume)\(', c) for c in cells): probs.append('needed a redo')
    if not any('Clean round' in o for o in outs): probs.append('no clean score')
    if (bad := [f for f in forbid if f in visible]): probs.append(f"script leaked into visible text: {', '.join(bad)}")
    return probs

A minimal clean round passes. Only a bare `dojo_start()` counts as dealing a round: `dojo_start('ab12')` replays a receipt and deals nothing, so it never affects the count.

In [ ]:
boot = ['doc(clik, pysk, edsk)', 'doc(dsk, exh, rgsk)', 'doc(acp)', 'list_pyskills()']
round_cells = ['dojo_start()', '%cd /tmp/kata', '# kata 1', 'doc(report.daily_report)', "dojo_score(bash_calls=0, report='RB7034')"]
outs = ['...', 'Clean round. Completion id: ab12']
test_eq(round_gates(boot+round_cells, outs, 'OK ready.'), [])
test_eq(round_gates(["dojo_start('ab12')"]+boot+round_cells, outs, 'OK ready.'), [])

Each gate names its own failure: a redealt round, a rescore, a redo, a missing clean receipt, and script leakage are separate problems, reported together.

In [ ]:
test_eq(round_gates(['dojo_start()']+round_cells, outs, ''), ['2 dojo_start calls'])
test_eq(round_gates(round_cells+['dojo_score(1)'], outs, ''), ['2 dojo_score calls (a clean round scores once, first try)'])
test_eq(round_gates(round_cells+['dojo_redo(3)'], outs, ''), ['needed a redo'])
test_eq(round_gates(round_cells, ['...'], ''), ['no clean score'])
round_gates(round_cells, outs, 'as these notes say')

## The round's shape

A captured dialog has structural boundaries that need no sentinels or bookkeeping: the bootstrap `doc()` reads, the `dojo_start()` that deals the round, and the score that closes it. `capture_slice` finds all three in a list of kernel cell sources - the run of bootstrap reads immediately before the last dealt round, that round's start, and its first score - so an abandoned earlier round in the same conversation is skipped, unrelated earlier calls are not bootstrap, and each backend only maps its native call/result pairs to cell sources before slicing. The split at the start index is what makes two prompt messages of one capture: the bootstrap reply and the round reply. `boot_gates` is the bootstrap's own acceptance test: nothing but doc reads and catalog calls.

In [ ]:
#| export
_BOOT_RE = r'^(doc\([^()\n]*\)|list_pyskills\(\))\s*$'

def capture_slice(
    cells, # Kernel cell sources of a conversation's calls, in order
):
    "The `(boot, start, last)` cell indices of a captured dialog: the bootstrap reads before the last dealt round, that round's `dojo_start()`, and its first score"
    starts = [i for i,c in enumerate(cells) if re.match(_START_RE, c)]
    if not starts: raise ValueError('no dojo_start() call')
    start = boot = starts[-1]
    while boot and re.match(_BOOT_RE, cells[boot-1].strip()): boot -= 1
    if boot == start: raise ValueError('no bootstrap doc call before dojo_start()')
    scores = [i for i,c in enumerate(cells[start:], start) if re.match(r'^dojo_score\(', c)]
    if not scores: raise ValueError('no dojo_score() after dojo_start()')
    return boot, start, scores[0]

def boot_gates(
    cells, # A bootstrap reply's kernel cell sources
):
    "The content problems that disqualify a bootstrap reply: a cell that is not a doc read or catalog call, or no doc read at all"
    probs = []
    if (bad := [c for c in cells if not re.match(_BOOT_RE, c.strip())]): probs.append(f'not a bootstrap cell: {bad[0]!r}')
    if not any(c.strip().startswith('doc(') for c in cells): probs.append('no doc() read')
    return probs

In [ ]:
test_eq(capture_slice(boot+round_cells), (0, 4, 8))
test_eq(capture_slice(['x = 1']+boot+round_cells), (1, 5, 9))   # earlier unrelated calls are not bootstrap
test_eq(capture_slice(boot+round_cells+boot+round_cells), (9, 13, 17))   # an abandoned round is skipped
test_fail(lambda: capture_slice(boot), contains='no dojo_start')
test_fail(lambda: capture_slice(round_cells), contains='no bootstrap doc call')
test_fail(lambda: capture_slice(boot+['dojo_start()']), contains='no dojo_score')
test_eq(boot_gates(boot), [])
test_eq(boot_gates(['doc(pysk)', 'x = 1']), ["not a bootstrap cell: 'x = 1'"])
test_eq(boot_gates(['list_pyskills()']), ['no doc() read'])

Both hosts also need the opposite selection when compacting: the turns a session should *keep*. A dojo round occupies whole user turns, so recognizing one is turn-level structure shared by both launchers: `_turns` chunks any item or record sequence at its user prompts, and each host supplies its own prompt test and its own strip.

In [ ]:
#| export
def _turns(
    xs, # Items or records in conversation order
    isprompt, # Is this element a real user prompt?
):
    "`xs` split into user turns at each prompt (a leading group holds anything earlier)"
    xs = L(xs)
    edges = L([0, *xs.argwhere(isprompt), len(xs)])
    return L(xs[s:e] for s,e in edges.pairwise())

In [ ]:
turns = _turns(['a','p1','x','y','p2','z'], lambda x: x.startswith('p'))
test_eq(turns, [['a'],['p1','x','y'],['p2','z']])
turns

The names a template claims to have documented are readable from the round itself: every standalone `doc(...)` call in its cells, expanded to both spellings a call site might use. Deriving them from the cells means a template built from any source - a headless capture, a live session, a replay - carries the right doc-state with no side channel.

In [ ]:
#| export
def doced_names(
    cells, # Kernel cell sources
):
    "Names documented by standalone `doc(...)` calls in `cells`, in both spellings"
    docs = set()
    for c in cells:
        for m in re.finditer(r'(?m)^\s*doc\(([^()\n]+)\)\s*$', c):
            for name in m.group(1).split(','): docs.update(_docnames(name.strip()))
    return sorted(docs)

In [ ]:
doced_names(['doc(find_msgs, view_dlg)', 'x = 1', "doc(report.daily_report)\nreport.daily_report(SAMPLE)"])

A template is assembled from two dialogs: the host's bootstrap (one prompt, its reply the startup's doc reads) and the shared round. `assemble` concatenates them and re-derives `doced` over every cell; `split_template` is the inverse a capture uses to write the two review files from one two-prompt dialog. Both work on the parsed shape `_msg_cells` gives for a prompt message: its reply exploded into note and code messages, and the kernel cell source of each call.

In [ ]:
#| export
def _pmsgs(dlg): return [m for m in dlg.messages if m.msg_type=='prompt']

def _msg_cells(
    pmsg, # A template prompt message
):
    "`(sub, codes, cells)`: the reply exploded, its code messages, and their kernel cell sources"
    sub = reply2dlg(pmsg)
    codes = [m for m in sub.messages if m.msg_type=='code']
    calls = [_parse_call(m.content) for m in codes]
    if bad := first(m.content for m,c in zip(codes,calls) if not (c and c[1].get('code'))): raise ValueError(f'not a kernel call: {bad}')
    return sub, codes, [c[1]['code'] for c in calls]

def template_cells(
    dlg, # A template dialog
):
    "Kernel cell sources of every prompt's reply in `dlg`, in order"
    return [c for m in _pmsgs(dlg) for c in _msg_cells(m)[2]]

def _one_prompt(m, name):
    "A one-prompt dialog holding a copy of `m`, with `doced` from its own cells"
    d = Dialog(name=name)
    d.mk_message(m.content, msg_type='prompt', output=m.output)
    d.meta['llmdojo'] = dict(doced=doced_names(template_cells(d)))
    return d

def assemble(
    boot, # The host's bootstrap dialog, or its path
    rnd=None, # The shared round dialog, or its path; the packaged round if None
):
    "The template a host splices: the bootstrap's prompt(s), then the round's, with `doced` re-derived over both"
    boot,rnd = [x if isinstance(x, Dialog) else read_ipynb(str(x)) for x in (boot, rnd or ROUND_PATH)]
    d = Dialog(name='dojo_template')
    for m in _pmsgs(boot)+_pmsgs(rnd): d.mk_message(m.content, msg_type='prompt', output=m.output)
    d.meta['llmdojo'] = dict(doced=doced_names(template_cells(d)))
    return d

def split_template(
    dlg, # A two-prompt template dialog: bootstrap, then round
):
    "`(boot, rnd)`: one-prompt dialogs for the bootstrap and the round, each with its own `doced`"
    b,r = _pmsgs(dlg)
    return _one_prompt(b, 'boot'), _one_prompt(r, 'dojo_round')

Assembling the packaged pair gives the two-prompt template; splitting it gives the pair back, and `doced` partitions the same way the cells do:

In [ ]:
tmpl = assemble(CLIK_BOOT)
test_eq([m.content for m in tmpl.messages], [BOOT_PROMPT, TMPL_PROMPT])
b,r = split_template(tmpl)
test_eq(b.messages[0].output, read_ipynb(str(CLIK_BOOT)).messages[0].output)
test_eq(sorted(b.meta['llmdojo']['doced']+r.meta['llmdojo']['doced']), tmpl.meta['llmdojo']['doced'])
test_eq(boot_gates(template_cells(b)), [])
tmpl.meta

## Refreshing a template

The round's cells rarely change, but their outputs go stale whenever the tooling's docs, card, or rendering move. Replaying the cells regenerates every receipt without model spend: a fresh `clikernel` CLI process (the same kernel, startup, magics, and rendering a live session gets) runs each cell in order in a scratch project, and the fresh outputs splice back into the template dialog. The kernel runs against the real machine state deliberately - the round deals into this machine's fixed run dir, and the replayed score's content-derived completion id is a registration we want. Stored templates never show that machine-specific path, though: on the way in, the canonical `/tmp/dojo` spelling is localized to the real run dir, and on the way out every occurrence maps back, so the committed artifact reads identically on every machine and refreshing an unchanged round on any box is diff-free (given the same installed tooling - environment-dependent outputs like the pyskill catalog still reflect the machine). Only the outputs move - cells and narration are inputs, so a change to the round itself is still capture-or-curation work.

In [ ]:
#| export
def _replay_cells(
    cells, # Kernel cell sources, run in order
    cwd, # Directory to start the kernel in
    env=None, # Extra environment entries for the kernel process
):
    "Each cell's rendered response from a fresh `clikernel` CLI at `cwd`"
    p = subprocess.Popen(['clikernel'], cwd=str(cwd), env=os.environ|(env or {}), text=True,
        stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=subprocess.DEVNULL)
    def upto(stop):
        res = []
        for line in p.stdout:
            if (line := line.rstrip('\n')) == stop: return res
            res.append(line)
        raise RuntimeError(f'clikernel exited during replay; last lines: {res[-5:]}')
    upto(_CLIK_MARKER)
    delim = p.stdout.readline().rstrip('\n')
    outs = []
    for c in cells:
        p.stdin.write(f'--\n{c}\n{delim}\n')
        p.stdin.flush()
        if (ack := p.stdout.readline().rstrip('\n')) != '.': raise RuntimeError(f'expected ack, got {ack!r}')
        outs.append('\n'.join(upto(delim)))
    p.stdin.write('exit\n')
    p.stdin.flush()
    p.wait()
    return outs

In [ ]:
rd = Path(tempfile.mkdtemp())
_replay_cells(['1+1', "print('hi')", 'x = 3'], rd)

`refresh_dialog` is the splice: given each prompt's fresh outputs (from whichever replayer the host owns), it writes them into the baked calls (an empty response renders as the host's no-output line, matching what a live session records), optionally renames the calls to the host's tool, re-derives `doced`, and gates - `round_gates` on the prompt that deals a round, `boot_gates` on any other - before canonicalizing. `refresh_template` is the clikernel-hosted whole: parse a packaged dialog's cells, replay them through a fresh `clikernel`, splice, write. It refuses a dialog whose calls aren't all kernel cells, and a refresh that fails the gates writes nothing - if the round no longer plays clean, the cells need human attention, not fresher outputs.

In [ ]:
#| export
DOJO_CANON = '/tmp/dojo'

def canon_tmpl(
    dlg, # A template dialog
    canon=DOJO_CANON, # Canonical spelling for the round's run-dir path
):
    "Rewrite the round's run-dir path (read from its own `%cd` cell) to `canon` throughout every reply, so stored templates read the same on every machine"
    rd = first(re.findall(r"%cd ([^\s'\"\\]+)", ' '.join(m.ai_res for m in _pmsgs(dlg))))
    if rd and rd != canon:
        for m in _pmsgs(dlg): m.output = prompt_output(m.ai_res.replace(rd, canon))
    return dlg

def localized_cells(
    dlg, # A template dialog
):
    "Per prompt, the reply's kernel cells with the canonical run dir localized to this machine's, ready to replay"
    from llmdojo.dojo import _run_dir
    return [[c.replace(DOJO_CANON, str(_run_dir())) for c in _msg_cells(m)[2]] for m in _pmsgs(dlg)]

def refresh_dialog(
    dlg, # A template dialog
    outs, # Per prompt, each cell's fresh output text, as `localized_cells` orders them
    tool=None, # Rename every call to this tool name (a host whose tool is not clikernel's)
):
    "Splice fresh outputs into `dlg`'s baked calls, gate each reply, re-derive `doced`, and canonicalize; returns `dlg`"
    from llmdojo.dojo import _run_dir
    for pm,o in zip(_pmsgs(dlg), outs):
        sub,codes,cells = _msg_cells(pm)
        for m,c,x in zip(codes, cells, o):
            name = tool or _parse_call(m.content)[0]
            if tool: m.content = f'{tool}(code={c.replace(str(_run_dir()), DOJO_CANON)!r})'
            m.output = code_output(x or f'({name} completed with no output)')
        notes = ' '.join(m.content for m in sub.messages if m.msg_type=='note')
        probs = round_gates(cells, o, notes) if any(re.match(_START_RE, c) for c in cells) else boot_gates(cells)
        if probs: raise ValueError('; '.join(probs))
        pm.output = prompt_output(dlg2reply(sub).replace(str(_run_dir()), DOJO_CANON))
    dlg.meta['llmdojo'] = dict(doced=doced_names(template_cells(dlg)))
    return dlg

def refresh_template(
    src, # Path to a template dialog .ipynb
    dst=None, # Where to write the refreshed dialog; `src` if None
):
    "Replay `src`'s kernel cells through a fresh clikernel, splice in current outputs, refresh `doced`, and gate; writes and returns the refreshed dialog"
    dlg = read_ipynb(str(src))
    with tempfile.TemporaryDirectory(prefix='dojorefresh_') as td:
        proj = Path(td)
        (proj/'pyproject.toml').write_text('[project]\nname = "dojo-refresh"\nversion = "0"\n')
        outs = [_replay_cells(cells, proj) for cells in localized_cells(dlg)]
    refresh_dialog(dlg, outs)
    write_ipynb(dlg, str(dst or src))
    return dlg

The acceptance test replays the packaged round twice and demands byte-identical results - full-pipeline determinism with zero model spend. It runs against real machine state by design (see above): the completion id it registers is content-derived, so repeated runs re-register the same receipt.

In [ ]:
td = Path(tempfile.mkdtemp())
r1 = refresh_template(ROUND_PATH, td/'r1.ipynb')
r2 = refresh_template(ROUND_PATH, td/'r2.ipynb')
test_eq((td/'r1.ipynb').read_bytes(), (td/'r2.ipynb').read_bytes())
test_eq(r1.meta['llmdojo']['doced'], read_ipynb(str(ROUND_PATH)).meta['llmdojo']['doced'])
assert DOJO_CANON in r1.messages[0].ai_res
b1 = refresh_template(CLIK_BOOT, td/'b1.ipynb')
test_eq(b1.meta['llmdojo']['doced'], read_ipynb(str(CLIK_BOOT)).meta['llmdojo']['doced'])

In [ ]:
from llmdojo.dojo import _run_dir

In [ ]:
assert str(_run_dir()) not in r1.messages[0].ai_res
shutil.rmtree(td)
len(r1.messages)

## Launch config

Each launcher runs its host tool itself rather than printing an id for command substitution: a config file can then supply standing arguments, such as the team's system-prompt files, which `$(claudedojo)` could never carry. `launch_config` reads the `<app>_args` list from `$XDG_CONFIG_HOME/<app>dojo/config.toml`, expanding `~` in each entry so the file can name prompt files portably.

In [ ]:
#| export
def launch_config(
    app, # Host tool name ('claude' or 'codex'): config at `$XDG_CONFIG_HOME/<app>dojo/config.toml`
    cfg=None, # Config file path, overriding the `app` default
):
    "Extra `app` args from the config file's `<app>_args` list, each `~`-expanded"
    if cfg is None: cfg = Path(os.environ.get('XDG_CONFIG_HOME') or '~/.config').expanduser()/f'{app}dojo'/'config.toml'
    if not Path(cfg).exists(): return []
    return [os.path.expanduser(o) for o in tomllib.loads(Path(cfg).read_text()).get(f'{app}_args', [])]

No config file means no extra args, so a bare install launches its tool with only the resume arguments:

In [ ]:
cdir = Path(tempfile.mkdtemp())
test_eq(launch_config('claude', cdir/'config.toml'), [])
(cdir/'config.toml').write_text('claude_args = ["--system-prompt-file", "~/prompts/sysp.md"]')
args = launch_config('claude', cdir/'config.toml')
assert '~' not in args[1]
args

## The dojobuild CLI

Template maintenance is one command, matching the layering: the launchers launch (and capture, which is backend-specific), while `dojobuild` owns the dialogs-to-stores pipeline. Bare `dojobuild` is the common case - refresh the two packaged clikernel dialogs (the shared round and the clikernel bootstrap), then build every store; chaining is safe because a refresh only regenerates machine-true receipts and `refresh_template` gates before writing. `--claude`/`--codex`/`--ipyai` build one store without refreshing: the post-review step after a capture or hand-curation. The ipyai build always replays, since its bootstrap is its own dialog and its tool renders through ipyai.

In [ ]:
#| export
@call_parse
def main(
    claude:bool=False, # Build only the Claude store, without refreshing
    codex:bool=False, # Build only the Codex store, without refreshing
    ipyai:bool=False, # Build only the ipyai store (its bootstrap and the round replayed through ipyai's `py`), without refreshing
):
    "Refresh the packaged clikernel dialogs (the shared round and the clikernel bootstrap, replayed through a fresh clikernel), then build every store"
    from llmdojo import claudedojo, codexdojo, ipyaidojo
    builds = dict(claude=claudedojo, codex=codexdojo, ipyai=ipyaidojo)
    only = [n for n,f in dict(claude=claude, codex=codex, ipyai=ipyai).items() if f]
    if not only:
        for p in (ROUND_PATH, CLIK_BOOT):
            refresh_template(p)
            print(f'refreshed: {p}')
    for n in only or builds:
        b = builds[n]
        r = b.build_template()
        if n == 'ipyai': asyncio.run(r)   # the ipyai replay is async (it drives a live kernel through ipyai's own client)
        print(f'built: {n} store ({b.TMPL_DIR})')

## Export -

In [ ]:
#|hide
#|eval: false
import nbdev; nbdev.nbdev_export()